# 28 — Text, Unicode, Characters, Words, and Bytes

**Master syllabus coverage:** V2 4.1–4.4

## Why this matters

A tokenizer cannot be correct without an explicit text representation. Unicode normalization and vocabulary units determine reversibility, coverage, sequence length, and what distinctions the SLM can learn.

## Learning objectives

- Distinguish graphemes, code points, encoded bytes, tokens, and token IDs.
- Compare canonical and compatibility normalization policies.
- Measure character, word, and byte vocabulary tradeoffs.
- Specify and test strict round-trip behavior.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Text is Unicode, storage is bytes, models consume IDs

A Python string is a sequence of Unicode code points. UTF-8 encodes those code points
into one to four bytes. What a human sees as one grapheme can contain multiple code
points (base character, combining mark, emoji joiner sequence). Tokenization must define
exactly which representation it accepts and preserve a reliable decode path.


In [ ]:
import re
import unicodedata

examples = ["café", "café", "😂", "👩‍💻", "東京"]
for text in examples:
    print(repr(text), "code points=", len(text), "utf8 bytes=", len(text.encode("utf-8")),
          "hex=", text.encode("utf-8").hex())
assert examples[0] != examples[1]
assert unicodedata.normalize("NFC", examples[0]) == unicodedata.normalize("NFC", examples[1])


## 2. Normalization is a product decision

NFC composes canonical sequences; NFD decomposes them. NFKC/NFKD additionally collapse
compatibility distinctions and can change meaning or style. Lowercasing, whitespace
normalization, and punctuation replacement likewise trade vocabulary size against lost
information. Store the exact policy with the tokenizer.


In [ ]:
samples = ["ＡＩ", "①", "Straße", "  timing…  matters  "]
for text in samples:
    print("original:", repr(text), "NFC:", repr(unicodedata.normalize("NFC", text)),
          "NFKC:", repr(unicodedata.normalize("NFKC", text)))

def conservative_normalize(text: str) -> str:
    return unicodedata.normalize("NFC", text).replace("\r\n", "\n")

assert conservative_normalize("café") == "café"


## 3. Character, word, and byte vocabularies

Character tokens are simple but Unicode makes coverage large and graphemes complicated.
Word tokens are interpretable but explode vocabulary and cannot naturally represent new
words. Bytes guarantee coverage with at most 256 base values, but sequences are longer
and individual bytes may not decode independently.


In [ ]:
corpus = ["jokes need timing", "timing needs pauses", "😂 needs context"]

characters = sorted(set("".join(corpus)))
words = sorted(set(token for line in corpus for token in re.findall(r"\w+|[^\w\s]", line)))
byte_values = sorted(set("\n".join(corpus).encode("utf-8")))
print("character vocab/length:", len(characters), sum(map(len, corpus)))
print("word vocab/length:", len(words), sum(len(re.findall(r"\w+|[^\w\s]", line)) for line in corpus))
print("observed byte vocab/length:", len(byte_values), len("\n".join(corpus).encode("utf-8")))


## 4. Round-trip invariants

For supported input, `decode(encode(text))` should equal the tokenizer's documented
normalized form. Unknown-token fallback is lossy; byte fallback preserves coverage.
Test empty strings, every special token, multilingual text, combining marks, newlines,
and malformed byte sequences according to explicit policy.


In [ ]:
def byte_encode(text: str) -> list[int]:
    return list(conservative_normalize(text).encode("utf-8"))

def byte_decode(ids: list[int]) -> str:
    if any(not 0 <= value <= 255 for value in ids):
        raise ValueError("byte token outside [0, 255]")
    return bytes(ids).decode("utf-8", errors="strict")

tests = ["", "hello", "café", "東京 😂\nnext line", "👩‍💻"]
for text in tests:
    expected = conservative_normalize(text)
    actual = byte_decode(byte_encode(text))
    assert actual == expected
print("all byte round trips passed")


## Failure modes to recognize

- **Normalization drift:** training and inference transform equivalent text differently.
- **Unknown-token loss:** unsupported text cannot be reconstructed.
- **Byte boundary confusion:** arbitrary token slices decode as invalid UTF-8.
- **Grapheme assumption:** code-point length is presented as user-visible character length.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Inspect code points and bytes for five examples from languages you expect Humor Machine to handle.
2. Write a normalization policy and list one useful distinction every transformation could erase.
3. Create a round-trip test table containing empty, whitespace, special-looking, multilingual, and emoji text.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can state exactly how raw text becomes bytes or symbols and prove supported text round-trips under a documented policy.

**Next:** 29 — Learned n-gram language models.
